In [2]:
import pandas as pd
from io import StringIO
import gzip

file_path = "20260207_20260207_Rexburg_Idaho.siz"  # <-- change to your .siz file name

def read_aeronet_siz(path: str) -> pd.DataFrame:
    # 1) Read raw text (handles plain or gzipped)
    if path.lower().endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
            text = f.read()
    else:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            text = f.read()

    # 2) Find the real CSV header line (starts with "Site,Date")
    lines = text.splitlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.startswith("Site,Date"):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Couldn't find the header line starting with 'Site,Date'.")

    # 3) Keep only header + data lines
    csv_text = "\n".join(lines[header_idx:])

    # 4) Load into DataFrame
    df = pd.read_csv(StringIO(csv_text))

    # 5) Clean columns a bit (optional)
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(" ", "_")
    )

    # 6) Convert what we can to numbers
    df = df.apply(pd.to_numeric, errors="ignore")

    # 7) Make a datetime column (super useful)
    date_col = "Date(dd:mm:yyyy)"
    time_col = "Time(hh:mm:ss)"
    if date_col in df.columns and time_col in df.columns:
        df["Datetime"] = pd.to_datetime(
            df[date_col].astype(str) + " " + df[time_col].astype(str),
            format="%d:%m:%Y %H:%M:%S",
            errors="coerce"
        )

    return df

df = read_aeronet_siz(file_path)

print(df.shape)
print(df.head(3))

# Optional: save to CSV
df.to_csv("20260207_Rexburg_Idaho.csv", index=False)

(2, 64)
            Site Date(dd:mm:yyyy) Time(hh:mm:ss)  Day_of_Year  \
0  Rexburg_Idaho       07:02:2026       19:52:50           38   
1  Rexburg_Idaho       07:02:2026       20:52:55           38   

   Day_of_Year(Fraction)  0.050000  0.065604  0.086077  0.112939  0.148184  \
0              38.828356  0.005845  0.014509  0.014181  0.007822  0.004527   
1              38.870081  0.001566  0.008939  0.015471  0.010924  0.005977   

   ...  If_AOD_is_L2  Last_Processing_Date(dd:mm:yyyy)  \
0  ...             0                        11:02:2026   
1  ...             0                        11:02:2026   

   Last_Processing_Time(hh:mm:ss)  Instrument_Number  Latitude(Degrees)  \
0                        15:27:14               1468           43.82011   
1                        15:29:37               1468           43.82011   

   Longitude(Degrees)  Elevation(m)  Inversion_Data_Quality_Level  \
0          -111.78301        1501.0                         lev15   
1          -111.78301 

/tmp/ipykernel_29153/1424435168.py:40: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(pd.to_numeric, errors="ignore")
